# Who is the best footballer of the last 25 years?

Most arguments about the greatest player are not disagreements about facts. They
are disagreements about **what the question means**, conducted as though they
were about facts. One person means peak. Another means longevity. A third means
trophies. Nobody says which, so nobody can be wrong, and nobody moves.

This project takes a different route. Instead of arguing about the answer, it
writes down **what greatness requires** — eleven things a player must have — and
then asks who has all of them.

A player who fails any single requirement is, by this definition, not the best.
And which requirement they failed is published, so the definition can be argued
with rather than simply asserted.

In [ ]:
import warnings

import matplotlib
import pandas as pd

from gambeta import needs, tifo

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")
keepers = pd.read_parquet(f"{SAMPLE}/keeper_ranking.parquet")

first = tifo.season_label(seasons["season"].min())
last = tifo.season_label(seasons["season"].max())

print(f"{len(ranking):,} players ranked")
print(f"{len(seasons):,} player-seasons")
print(f"leagues: {', '.join(sorted(seasons['league'].unique()))}")
print(f"seasons: {first} to {last}")

## The eleven requirements

Each is measured across every season and every league, then pooled per player
weighted by minutes. All are oriented so that **higher is better** — including
discipline, which is negated at source so a clean player scores above a dirty
one without any downstream code needing to know the direction.

In [ ]:
pd.DataFrame(
    [
        {"#": i + 1, "requirement": r.label, "key": r.key, "grain": r.kind}
        for i, r in enumerate(needs.OUTFIELD)
    ]
).set_index("#")

## Gate first, then rank

"Must have" is read literally. A player has to clear the 40th percentile on
**all eleven** requirements to qualify at all. Only then are qualifiers ranked.

A weighted average alone would let a player be genuinely poor at something the
list calls mandatory and still win on volume elsewhere. The gate is what stops
that.

In [ ]:
qualified = ranking[ranking["qualified"]]
print(
    f"{len(qualified)} of {len(ranking):,} players clear all eleven requirements "
    f"({100 * len(qualified) / len(ranking):.1f}%)"
)

qualified.head(15)[["player", "score", "seasons", "leagues"]].round(2)

In [ ]:
fig = tifo.ranked_dots(
    qualified.assign(lo=qualified["score"], hi=qualified["score"]),
    top=20,
    title="Who has all eleven",
    subtitle="Composite of eleven requirements, era- and league-adjusted.",
)

## A career is not league-bound

Look at the `leagues` column. Ronaldo's career is England, Spain and Italy —
nineteen seasons pooled into one player, not three separate league careers.

This is the single most important structural decision in the project. Ranking
"the best player in a league" answers a different and much less interesting
question, and it makes players who moved impossible to evaluate.

In [ ]:
multi = qualified[qualified["leagues"].str.contains(",")]
print(f"{len(multi)} of the {len(qualified)} qualifiers played in more than one league\n")
multi.head(10)[["player", "score", "seasons", "leagues"]].round(2)

## Who fails, and on what

This is the part most rankings hide. A great player missing the cut is more
informative than one making it, because it tells you exactly where the
definition bites.

In [ ]:
near = ranking[~ranking["qualified"]].copy()
near["missed"] = near["failed"].fillna("").apply(lambda s: len(s.split(", ")) if s else 0)

one = near[near["missed"] == 1].nlargest(12, "score")
one[["player", "score", "seasons", "failed"]].round(2)

In [ ]:
pd.read_csv(f"{SAMPLE}/failures.csv").head(11)

## Goalkeepers, judged on their own job

Keepers score zero on goals and assists, so under an attacking rating they rank
near the bottom — meaninglessly. They get a parallel list of eight requirements
built from save percentage, clean sheets and goals conceded.

They are **not** claimed to be comparable with outfielders. Two leaderboards, no
combined number, because the data cannot support one.

In [ ]:
kq = keepers[keepers["qualified"]]
print(f"{len(kq)} of {len(keepers)} keepers qualify\n")
kq.head(10)[["player", "score", "seasons", "leagues"]].round(2)

## What would change my mind

A conclusion is worth what its author would abandon it for. Mine:

1. **Defensive contribution.** FBref records no per-player defensive action
   before 2017-18, so a centre-back is invisible to requirements 1–6. This is an
   attacking-contribution rating and is named as such. Real defensive data would
   change the list completely.
2. **The gate height.** The 40th percentile is a judgement call. Raising it to 50
   would disqualify players currently sitting just inside.
3. **The weights.** Every requirement counts equally right now. Anyone who
   thinks scoring matters more than availability can say so numerically, and the
   ranking will move.
4. **Missing football.** No Champions League, no internationals, no Ligue 1, and
   nothing outside the Big 5. Careers at Sporting, Al-Nassr or Inter Miami are
   invisible.

The productive question is not "is this wrong" but **"which of these four would
you change, and to what"** — an argument that can actually make progress.